**Our research question**

Can a team-level model determine whether the current starter at a position is worth keeping relative to positional peers and their cost?

In [2]:
# Our imports!
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import nflreadpy as nfl

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, precision_score, recall_score, f1_score,
)

In [7]:
# Play-by-play data from nflverse (via nflreadpy), 2015-2024
seasons = list(range(2015, 2025))

columns_we_need = [
    # Identify plays, seasons, and teams
    "season", "season_type", "week", "game_id", "play_id",
    "posteam", "defteam",

    # Identify players
    "passer_id", "passer_player_id", "passer_player_name",
    "rusher_player_id", "rusher_player_name",
    "receiver_player_id", "receiver_player_name",

    # Game situation
    "down", "ydstogo", "yardline_100",
    "qtr", "game_seconds_remaining", "score_differential",

    # Play eligibility and type
    "play_type", "play_deleted", "qb_dropback",
    "pass_attempt", "rush_attempt", "qb_scramble",
    "qb_kneel", "qb_spike", "two_point_attempt",

    # Performance
    "epa", "qb_epa", "success", "cpoe",
    "passing_yards", "rushing_yards", "receiving_yards",
    "complete_pass", "pass_touchdown", "rush_touchdown",
    "interception", "sack", "air_yards", "yards_after_catch",
    "first_down_pass", "first_down_rush",

    # Fumble attribution
    "fumble_lost", "fumbled_1_player_id", "fumbled_1_team",
    "fumbled_2_player_id", "fumbled_2_team",
]

pbp = nfl.load_pbp(seasons).select(columns_we_need).to_pandas()

print(pbp.shape)
pbp.head()

(483605, 50)


,season,season_type,week,game_id,play_id,posteam,defteam,passer_id,passer_player_id,passer_player_name,...,sack,air_yards,yards_after_catch,first_down_pass,first_down_rush,fumble_lost,fumbled_1_player_id,fumbled_1_team,fumbled_2_player_id,fumbled_2_team
0,2015,REG,1,2015_01_BAL_DEN,1.0,None,None,None,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,None,None,None,None
1,2015,REG,1,2015_01_BAL_DEN,36.0,BAL,DEN,None,None,None,...,0.0,NaN,NaN,0.0,0.0,0.0,None,None,None,None
2,2015,REG,1,2015_01_BAL_DEN,51.0,BAL,DEN,00-0026158,00-0026158,J.Flacco,...,0.0,3.0,0.0,0.0,0.0,0.0,None,None,None,None
3,2015,REG,1,2015_01_BAL_DEN,75.0,BAL,DEN,None,None,None,...,0.0,NaN,NaN,0.0,0.0,0.0,None,None,None,None
4,2015,REG,1,2015_01_BAL_DEN,96.0,BAL,DEN,00-0026158,00-0026158,J.Flacco,...,0.0,4.0,6.0,1.0,0.0,0.0,None,None,None,None


In [4]:
pbp.head(3)

,season,game_id,desc,down,play_type,fourth_down_converted,fourth_down_failed,ydstogo,yardline_100,qtr,game_seconds_remaining,score_differential,posteam_timeouts_remaining,wp,shotgun
0,2015,2015_01_BAL_DEN,GAME,NaN,None,NaN,NaN,0.0,NaN,1.0,3600.0,NaN,NaN,0.422024,0.0
1,2015,2015_01_BAL_DEN,8-B.McManus kicks 65 yards from DEN 35 to end ...,NaN,kickoff,0.0,0.0,0.0,35.0,1.0,3600.0,0.0,3.0,0.422024,0.0
2,2015,2015_01_BAL_DEN,(15:00) 5-J.Flacco pass short left to 44-K.Jus...,1.0,pass,0.0,0.0,10.0,80.0,1.0,3600.0,0.0,3.0,0.422024,0.0


# Getting salaries

- Get NFL player salary data for roughly the last 10 years so we can see whether a player’s performance is over- or undervalued relative to their pay, compared with others at the same position.
- NFL verse already has a contract feature from OverTheCap.com so we'll use that
- However each row is one player's contract, not contracts throughout the years. But the year by year salary is nested inside the "season_history" column that holds a list of dicts, one per column.
- We will chose between cap_number (the team’s cap hit that year) or cap_percent (cap hit as a share of that year’s salary cap). The percent version makes different years comparable, since the cap grows.
- Because the salaries are nested in season_history we must flatten out the data. 
    - This is done by:
        - explode gives each season its own row.
        - pd.json_normalize turns each dict into columns.
        - concat reattaches the player name and position

In [9]:
contracts = nfl.load_contracts().to_pandas()

print(f"Rows: {contracts.shape[0]:,}")
print(contracts.columns.tolist())
contracts.head()

Rows: 53,513
['player', 'position', 'team', 'is_active', 'year_signed', 'years', 'value', 'apy', 'guaranteed', 'apy_cap_pct', 'inflated_value', 'inflated_apy', 'inflated_guaranteed', 'player_page', 'otc_id', 'gsis_id', 'height', 'weight', 'college', 'draft_year', 'draft_round', 'draft_overall', 'draft_team', 'date_of_birth', 'season_history', 'contract_history']


,player,position,team,is_active,year_signed,years,value,apy,guaranteed,apy_cap_pct,...,height,weight,college,draft_year,draft_round,draft_overall,draft_team,date_of_birth,season_history,contract_history
0,Joe Burrow,QB,Bengals,True,2023,5.0,275.000,55.000000,146.510,0.245,...,"6'4""",215,LSU,2020.0,1.0,1.0,Bengals,None,"[{'year': '2020', 'team': 'Bengals', 'base_sal...","[{'team': 'Bengals', 'contract_type': 'Drafted..."
1,Aaron Rodgers,QB,NYJ/GB,False,2022,3.0,150.815,50.271667,101.415,0.241,...,"6'2""",225,California,2005.0,1.0,24.0,Packers,None,"[{'year': '2005', 'team': 'Packers', 'base_sal...","[{'team': 'Packers', 'contract_type': 'Drafted..."
2,Josh Allen,QB,Bills,False,2021,6.0,258.000,43.000000,100.000,0.236,...,"6'5""",233,Wyoming,2018.0,1.0,7.0,Bills,None,"[{'year': '2018', 'team': 'Bills', 'base_salar...","[{'team': 'Bills', 'contract_type': 'Drafted',..."
3,Russell Wilson,QB,Broncos,False,2022,5.0,245.000,49.000000,124.000,0.235,...,"5'11""",206,Wisconsin,2012.0,3.0,75.0,Seahawks,"November 29, 1988","[{'year': '2012', 'team': 'Seahawks', 'base_sa...","[{'team': 'Seahawks', 'contract_type': 'Drafte..."
4,Dak Prescott,QB,Cowboys,True,2024,4.0,240.000,60.000000,129.000,0.235,...,"6'2""",226,Mississippi State,2016.0,4.0,135.0,Cowboys,None,"[{'year': '2016', 'team': 'Cowboys', 'base_sal...","[{'team': 'Cowboys', 'contract_type': 'Drafted..."


In [10]:
# See what's inside the nested column for one player
burrow = contracts[contracts["player"] == "Joe Burrow"].iloc[0]
print(burrow["season_history"][0])   

{'year': '2020', 'team': 'Bengals', 'base_salary': 0.61, 'prorated_bonus': 5.970025, 'option_bonus': 0.0, 'roster_bonus': 0.0, 'guaranteed_salary': 0.61, 'cap_number': 6.580025, 'cap_percent': 0.032, 'cash_paid': 24.4901, 'workout_bonus': None, 'per_game_roster_bonus': None, 'other_bonus': None}


Looks like salary is in millions. We should use cap_number or cap_percent to see what a team spent on a given player. 

In [11]:
step1 = (
    contracts[
        ["gsis_id", "otc_id", "player", "position", "season_history"]
    ]
    .explode("season_history")
    .dropna(subset=["season_history"])
    .reset_index(drop=True)
)

detail = pd.json_normalize(step1["season_history"].tolist())

salary_df = pd.concat(
    [step1.drop(columns="season_history"), detail],
    axis=1,
)

# Removes non-year entries such as "Total"
salary_df["season"] = pd.to_numeric(
    salary_df["year"], errors="coerce"
)

salary_df = salary_df[
    salary_df["season"].isin(seasons)
].copy()

salary_df["season"] = salary_df["season"].astype(int)

salary_df = salary_df[
    [
        "gsis_id", "otc_id", "player", "position", "season",
        "team", "cap_number", "cap_percent", "cash_paid",
    ]
]

print(salary_df.shape)
salary_df.head()

(184136, 9)


,gsis_id,otc_id,player,position,season,team,cap_number,cap_percent,cash_paid
0,00-0036442,8741,Joe Burrow,QB,2020,Bengals,6.580025,0.032,24.490100
1,00-0036442,8741,Joe Burrow,QB,2021,Bengals,8.225035,0.042,2.255006
2,00-0036442,8741,Joe Burrow,QB,2022,Bengals,9.870037,0.046,3.900012
3,00-0036442,8741,Joe Burrow,QB,2023,Bengals,19.515043,0.086,45.545018
4,00-0036442,8741,Joe Burrow,QB,2024,Bengals,29.551112,0.112,65.714982


In [12]:
duplicates = salary_df.duplicated(
    subset=["gsis_id", "season", "team"],
    keep=False,
)

salary_df.loc[duplicates].head(20)

,gsis_id,otc_id,player,position,season,team,cap_number,cap_percent,cash_paid
0,00-0036442,8741,Joe Burrow,QB,2020,Bengals,6.580025,0.032,24.490100
1,00-0036442,8741,Joe Burrow,QB,2021,Bengals,8.225035,0.042,2.255006
2,00-0036442,8741,Joe Burrow,QB,2022,Bengals,9.870037,0.046,3.900012
3,00-0036442,8741,Joe Burrow,QB,2023,Bengals,19.515043,0.086,45.545018
4,00-0036442,8741,Joe Burrow,QB,2024,Bengals,29.551112,0.112,65.714982
24,00-0023459,1085,Aaron Rodgers,QB,2015,Packers,18.250000,0.121,11.600000
25,00-0023459,1085,Aaron Rodgers,QB,2016,Packers,19.250000,0.118,12.600000
26,00-0023459,1085,Aaron Rodgers,QB,2017,Packers,20.300000,0.116,13.312500
27,00-0023459,1085,Aaron Rodgers,QB,2018,Packers,20.900000,0.113,66.900000
28,00-0023459,1085,Aaron Rodgers,QB,2019,Packers,29.352000,0.149,29.260000
